# iSign English → Sign Pose — pre-training pipeline

This notebook takes the iSign dataset (already on Google Drive) and the Progressive Transformers SLP model and gets them to the point where **training can start**. It does **not** train anything.

**What it does, in order**

1. Checks the dataset on Drive (the Hugging Face → Drive transfer is kept below but commented out).
2. Clones the Progressive Transformers repository and applies a small set of *asserted* patches so it works on a current Colab (no `torchtext`, correct padding masks, loss that ignores padded frames).
3. Reads the split pose archive straight from Drive and converts a small pilot (train / dev / test) into the model's format: 75 landmarks × 3 values + 1 progress counter = **226 values per frame**.
4. Builds the model with `trg_size = 225` for that target format, using an English vocabulary built from the iSign text.
5. Runs a set of checks (shapes, masks, loss, gradients, padding invariance, greedy decoding) so problems show up here and not mid-training.

**Before you run:** use a Colab runtime (GPU optional — the checks also run on CPU) and run the cells top to bottom. If Colab asks you to restart the session after the `pip install` cell, restart and run again from the first cell.

**On "the model trained on a different dataset":** the Progressive Transformers repo ships no pretrained weights, so there is nothing to load. The architecture is built fresh with the iSign sizes (target width 226, iSign vocabulary). If you do have a checkpoint from another dataset, cell 14b shows how to load only the layers whose shapes match.

## 1. Setup and settings

In [ ]:
# CELL 1 — Mount Drive and define every setting used later in one place.
import bisect
import io
import itertools
import importlib
import os
import random
import re
import shutil
import subprocess
import sys
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

from google.colab import drive
drive.mount("/content/drive")

# ---------- Drive locations (the dataset is already uploaded here) ----------
DATA_DIR = Path("/content/drive/MyDrive/ISL_Project/iSign")
CSV_PATH = DATA_DIR / "iSign_v1.1.csv"
POSE_DIR = DATA_DIR / "pose_parts"
POSE_PART_NAMES = [f"iSign-poses_v1.1_part_{s}" for s in ("aa", "ab", "ac", "ad")]

# ---------- Local (Colab disk) locations ----------
REPO_URL = "https://github.com/BenSaunders27/ProgressiveTransformersSLP.git"
REPO_COMMIT_TESTED = "ff8b28cb0817d177f63dfa18c7c322806ee5dd96"   # the version these patches were written against
REPO_DIR = Path("/content/ProgressiveTransformersSLP")
WORK_DIR = Path("/content/isl_phase2_pilot")

# ---------- Optional: keep the patched code + pilot data on Drive (last cell) ----------
SAVE_TO_DRIVE = True
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/ISL_Project/pre_training_artifacts")

# ---------- Pilot size (sentences) ----------
N_TRAIN, N_DEV, N_TEST = 8, 2, 2
MIN_FRAMES, MAX_FRAMES = 2, 600     # sequences outside this range are skipped
PE_MAX_LEN = 5000                   # positional-encoding table length; must exceed MAX_FRAMES
BATCH_SIZE = 2

# ---------- Pose -> model-frame conversion ----------
N_BODY, N_HAND = 33, 21
N_LANDMARKS = N_BODY + 2 * N_HAND   # 75 landmarks kept: body + left hand + right hand
TRG_SIZE = N_LANDMARKS * 3          # 225 coordinates per frame (the counter makes it 226)
COORD_HALF_RANGE = 150.0            # x/y are assumed to lie in roughly [0, 300] -> [-1, 1]
CONF_THRESHOLD = 0.5                # landmarks below this confidence are zeroed
SEED = 27

assert MAX_FRAMES < PE_MAX_LEN, "PE_MAX_LEN must be larger than MAX_FRAMES"
print("Settings loaded.")

## 2. File transfer to Drive — already done, commented out
The cells below are the original Hugging Face → Drive transfer, kept verbatim but commented out. Select all lines and press `Ctrl+/` to re-enable them if you ever need to re-download.

In [ ]:
# CELLS 3-6 (original) — Hugging Face login and metadata download
# ALREADY DONE — the dataset is on Drive. Everything below is commented out on purpose;
# select all lines and press Ctrl+/ to re-enable. (Original code, unchanged.)

# # CELL 3 — Install/upgrade the Hugging Face Hub client.
# !pip install -U huggingface_hub
#

# # CELL 4 — Log in to Hugging Face so gated iSign files can be accessed.
# from huggingface_hub import notebook_login
# notebook_login()
#

# # CELL 5 — List files published in the iSign dataset repository.
#
# from huggingface_hub import HfApi
#
# api = HfApi()
# repo_id = "Exploration-Lab/iSign"
#
# info = api.dataset_info(
#     repo_id,
#     files_metadata=True
# )
#
# for f in info.siblings:
#     size = getattr(f, "size", None)
#     size_gb = f"{size / 1e9:.2f} GB" if size is not None else "size unknown"
#     print(f"{size_gb:>12} | {f.rfilename}")
#

# # CELL 6 — Download the iSign CSV metadata files to Google Drive.
# from huggingface_hub import hf_hub_download
# from pathlib import Path
# import shutil
#
# DRIVE_DIR = Path("/content/drive/MyDrive/ISL_Project/iSign")
# DRIVE_DIR.mkdir(parents=True, exist_ok=True)
#
# repo_id = "Exploration-Lab/iSign"
#
# files = [
#     "iSign_v1.1.csv",
#     "word-description-dataset_v1.1.csv",
#     "word-presence-dataset_v1.1.csv",
# ]
#
# for filename in files:
#     downloaded = hf_hub_download(
#         repo_id=repo_id,
#         filename=filename,
#         repo_type="dataset"
#     )
#
#     destination = DRIVE_DIR / filename
#     shutil.copy2(downloaded, destination)
#     print(f"Saved: {destination}")
#

In [ ]:
# CELL 10 (original) — transfer the large pose/video parts from Hugging Face to Drive
# ALREADY DONE — the dataset is on Drive. Everything below is commented out on purpose;
# select all lines and press Ctrl+/ to re-enable. (Original code, unchanged.)

# # CELL 10 — Transfer the large gated iSign pose/video parts from Hugging Face to Drive.
# # One-cell pipeline: download one file -> upload to Drive -> verify -> clear local cache -> next file
#
# from pathlib import Path
# import shutil
# import time
# from huggingface_hub import HfApi, hf_hub_download
# from google.colab import auth
# from googleapiclient.discovery import build
# from googleapiclient.http import MediaFileUpload
#
# REPO_ID = "Exploration-Lab/iSign"
# REPO_TYPE = "dataset"
# ROOT_NAME = "ISL_Project"
# PROJECT_NAME = "iSign"
# POSE_FOLDER_NAME = "pose_parts"
# VIDEO_FOLDER_NAME = "video_parts"
#
# POSE_FILES = [
#     "iSign-poses_v1.1_part_aa",
#     "iSign-poses_v1.1_part_ab",
#     "iSign-poses_v1.1_part_ac",
#     "iSign-poses_v1.1_part_ad",
# ]
# VIDEO_FILES = [
#     "iSign-videos_v1.1_part_aa",
#     "iSign-videos_v1.1_part_ab",
# ]
# FILES_TO_TRANSFER = [(name, POSE_FOLDER_NAME) for name in POSE_FILES] + [
#     (name, VIDEO_FOLDER_NAME) for name in VIDEO_FILES
# ]
#
# # Authenticate and initialize services. If already authenticated, Colab reuses the session.
# auth.authenticate_user()
# drive_service = build("drive", "v3")
# hf_api = HfApi()
#
# def find_folder(name, parent_id):
#     q = (
#         f"name = '{name}' and "
#         "mimeType = 'application/vnd.google-apps.folder' and "
#         "trashed = false"
#     )
#     if parent_id:
#         q += f" and '{parent_id}' in parents"
#     result = drive_service.files().list(
#         q=q, fields="files(id,name,parents)", pageSize=100
#     ).execute()
#     matches = result.get("files", [])
#     if matches:
#         return matches[0]["id"]
#     return None
#
# def create_folder(name, parent_id):
#     body = {
#         "name": name,
#         "mimeType": "application/vnd.google-apps.folder",
#         "parents": [parent_id],
#     }
#     result = drive_service.files().create(body=body, fields="id,name").execute()
#     print(f"Created Drive folder: {name}")
#     return result["id"]
#
# def get_or_create_folder(name, parent_id):
#     folder_id = find_folder(name, parent_id)
#     return folder_id if folder_id else create_folder(name, parent_id)
#
# def find_drive_file(name, folder_id):
#     safe_name = name.replace("'", r"\'")
#     q = (
#         f"name = '{safe_name}' and '{folder_id}' in parents "
#         "and trashed = false"
#     )
#     result = drive_service.files().list(
#         q=q, fields="files(id,name,size,mimeType)", pageSize=100
#     ).execute()
#     return result.get("files", [])
#
# def remove_cached_file(filename, downloaded_path=None):
#     """Remove only this file's local HF snapshot link/blob after Drive verification."""
#     cache_root = Path.home() / ".cache/huggingface/hub/datasets--Exploration-Lab--iSign"
#     candidates = set()
#
#     if downloaded_path:
#         p = Path(downloaded_path)
#         if p.exists() or p.is_symlink():
#             candidates.add(p)
#             try:
#                 resolved = p.resolve()
#                 if resolved != p:
#                     candidates.add(resolved)
#             except Exception:
#                 pass
#
#     if cache_root.exists():
#         for p in cache_root.rglob(filename):
#             if p.is_file() or p.is_symlink():
#                 candidates.add(p)
#                 try:
#                     resolved = p.resolve()
#                     if resolved != p and cache_root in resolved.parents:
#                         candidates.add(resolved)
#                 except Exception:
#                     pass
#
#     removed_bytes = 0
#     for p in sorted(candidates, key=lambda x: len(str(x)), reverse=True):
#         try:
#             if p.is_symlink():
#                 p.unlink(missing_ok=True)
#             elif p.is_file():
#                 size = p.stat().st_size
#                 p.unlink(missing_ok=True)
#                 removed_bytes += size
#         except FileNotFoundError:
#             pass
#         except Exception as exc:
#             print(f"Cache cleanup warning for {p}: {exc}")
#     if removed_bytes:
#         print(f"Freed about {removed_bytes / 1e9:.2f} GB from local cache.")
#     return removed_bytes
#
# # Pull authoritative file sizes from the dataset repository metadata.
# print("Reading dataset file metadata from Hugging Face...")
# repo_info = hf_api.dataset_info(REPO_ID, files_metadata=True)
# repo_sizes = {
#     item.rfilename: getattr(item, "size", None)
#     for item in repo_info.siblings
# }
# missing = [name for name, _ in FILES_TO_TRANSFER if name not in repo_sizes]
# if missing:
#     raise RuntimeError(
#         "Could not find these expected files in Hugging Face metadata: "
#         + ", ".join(missing)
#     )
#
# # Resolve folders from My Drive instead of hardcoding folder IDs.
# root_id = find_folder(ROOT_NAME, "root")
# if not root_id:
#     root_id = create_folder(ROOT_NAME, "root")
# project_id = get_or_create_folder(PROJECT_NAME, root_id)
# pose_folder_id = get_or_create_folder(POSE_FOLDER_NAME, project_id)
# video_folder_id = get_or_create_folder(VIDEO_FOLDER_NAME, project_id)
# folder_ids = {
#     POSE_FOLDER_NAME: pose_folder_id,
#     VIDEO_FOLDER_NAME: video_folder_id,
# }
#
# print("\nTransfer plan:")
# total_bytes = sum(int(repo_sizes[name] or 0) for name, _ in FILES_TO_TRANSFER)
# print(f"{len(FILES_TO_TRANSFER)} large files | metadata total ~{total_bytes / 1e9:.2f} GB")
# print("Destination: My Drive / ISL_Project / iSign / [pose_parts or video_parts]")
#
# for index, (filename, folder_name) in enumerate(FILES_TO_TRANSFER, start=1):
#     expected_size = repo_sizes.get(filename)
#     if expected_size is None:
#         raise RuntimeError(f"File size unavailable for {filename}; stopping safely.")
#     expected_size = int(expected_size)
#     folder_id = folder_ids[folder_name]
#
#     print(f"\n{'=' * 72}")
#     print(f"[{index}/{len(FILES_TO_TRANSFER)}] {filename}")
#     print(f"Expected size: {expected_size / 1e9:.2f} GB")
#
#     # Check Drive first. If the exact-size file is already there, don't upload it again.
#     existing_files = find_drive_file(filename, folder_id)
#     matching = next(
#         (f for f in existing_files if int(f.get("size", -1)) == expected_size),
#         None
#     )
#     if matching:
#         print(f"Already complete in Drive; skipping upload. Drive ID: {matching['id']}")
#         # It is safe to clear the local cache copy only after confirming Drive size.
#         remove_cached_file(filename)
#         free_gb = shutil.disk_usage("/").free / 1e9
#         print(f"Local free disk: {free_gb:.2f} GB")
#         continue
#
#     # Remove stale/incomplete same-name Drive entries before uploading a clean copy.
#     for old in existing_files:
#         print(
#             f"Removing mismatched Drive entry ({old.get('size', 'unknown')} bytes) "
#             f"before retry: {old['id']}"
#         )
#         drive_service.files().delete(fileId=old["id"]).execute()
#
#     # If not cached, make sure there's enough disk for this single file plus safety margin.
#     cache_root = Path.home() / ".cache/huggingface/hub/datasets--Exploration-Lab--iSign"
#     cached_candidates = [
#         p for p in cache_root.rglob(filename)
#         if p.is_file() and p.stat().st_size == expected_size
#     ] if cache_root.exists() else []
#
#     downloaded_path = str(cached_candidates[0]) if cached_candidates else None
#     if downloaded_path is None:
#         free_bytes = shutil.disk_usage("/").free
#         safety_margin = 6 * 1024**3
#         required = expected_size + safety_margin
#         if free_bytes < required:
#             raise RuntimeError(
#                 f"Not enough Colab disk space for {filename}. "
#                 f"Free: {free_bytes / 1e9:.2f} GB; "
#                 f"needed about {required / 1e9:.2f} GB. "
#                 "No new download was started. Ensure previous verified cache files "
#                 "were cleared, then rerun this cell."
#             )
#         print("Downloading this file to the local HF cache (one file at a time)...")
#         downloaded_path = hf_hub_download(
#             repo_id=REPO_ID,
#             filename=filename,
#             repo_type=REPO_TYPE,
#         )
#     else:
#         print("Using existing complete cached file; no new download needed.")
#
#     source = Path(downloaded_path)
#     actual_source_size = source.stat().st_size
#     if actual_source_size != expected_size:
#         raise RuntimeError(
#             f"Local file size mismatch for {filename}: "
#             f"{actual_source_size} != expected {expected_size}. "
#             "Keeping the local file for diagnosis."
#         )
#
#     print(f"Local source ready: {source}")
#     print(f"Uploading to Drive folder '{folder_name}'...")
#     media = MediaFileUpload(
#         str(source),
#         mimetype="application/octet-stream",
#         chunksize=8 * 1024 * 1024,
#         resumable=True,
#     )
#     request = drive_service.files().create(
#         body={"name": filename, "parents": [folder_id]},
#         media_body=media,
#         fields="id,name,size",
#     )
#
#     response = None
#     last_reported = -5
#     start_time = time.time()
#     while response is None:
#         status, response = request.next_chunk(num_retries=5)
#         if status:
#             percent = int(status.progress() * 100)
#             if percent >= last_reported + 5 or percent == 100:
#                 elapsed = max(time.time() - start_time, 1)
#                 sent_gb = expected_size * status.progress() / 1e9
#                 speed_mbps = (expected_size * status.progress() / elapsed) / 1e6
#                 print(
#                     f"Upload: {percent:3d}% | {sent_gb:.2f}/{expected_size / 1e9:.2f} GB "
#                     f"| average {speed_mbps:.1f} MB/s"
#                 )
#                 last_reported = percent
#
#     uploaded_id = response["id"]
#     uploaded_size = int(response.get("size", -1))
#
#     # Verify with a fresh Drive metadata lookup, not only the upload response.
#     verified = drive_service.files().get(
#         fileId=uploaded_id, fields="id,name,size,mimeType,parents"
#     ).execute()
#     drive_size = int(verified.get("size", -1))
#     if uploaded_size != expected_size or drive_size != expected_size:
#         raise RuntimeError(
#             f"Drive verification failed for {filename}. "
#             f"Expected {expected_size}; upload response {uploaded_size}; "
#             f"Drive metadata {drive_size}. Local cache has been retained."
#         )
#
#     print(f"VERIFIED: {filename} | {drive_size} bytes | Drive ID: {uploaded_id}")
#
#     # Only delete local cached bytes after exact-size Drive verification.
#     remove_cached_file(filename, downloaded_path)
#     print(f"Finished {filename}. Local free disk: {shutil.disk_usage('/').free / 1e9:.2f} GB")
#
# print("\nALL REQUESTED LARGE FILES ARE VERIFIED IN GOOGLE DRIVE.")
# print("Pose destination:", f"ISL_Project/iSign/{POSE_FOLDER_NAME}")
# print("Video destination:", f"ISL_Project/iSign/{VIDEO_FOLDER_NAME}")
#

## 3. Check the dataset on Drive

In [ ]:
# CELL 5 — Confirm the dataset is on Drive and inspect the CSV.
import pandas as pd

required = [CSV_PATH] + [POSE_DIR / n for n in POSE_PART_NAMES]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "These files are not on Drive (re-enable the transfer cells above):\n  " + "\n  ".join(missing)
    )
for p in required:
    print(f"{p.stat().st_size / 1e9:8.2f} GB  {p.name}")

df = pd.read_csv(CSV_PATH)
assert {"uid", "text"} <= set(df.columns), f"Unexpected CSV columns: {df.columns.tolist()}"
df["uid"] = df["uid"].astype(str)
df["video_id"] = df["uid"].str.rsplit("-", n=1).str[0]
print(f"\nText segments: {len(df):,} | unique videos: {df['video_id'].nunique():,}")
display(df[["uid", "text"]].head(5))

## 4. Install the pose reader and clone the model code

In [ ]:
# CELL 6 — Install pose-format (reads the iSign .pose files) and report versions.
# If Colab asks to restart the session after this cell, restart and run again from CELL 1.
%pip install -q pose-format

import importlib.metadata as md
import numpy as np
import torch

for pkg in ("pose-format", "numpy", "torch"):
    try:
        print(f"{pkg:12s}", md.version(pkg))
    except Exception:
        print(f"{pkg:12s} version unknown")
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

import pose_format  # fails here, rather than later, if the install did not work

In [ ]:
# CELL 7 — Clone a fresh copy of the Progressive Transformers repository.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)   # always start from a pristine copy so the patches below apply cleanly
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

head = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("Repository commit:", head)
if head != REPO_COMMIT_TESTED:
    print(f"WARNING: patches were written for {REPO_COMMIT_TESTED}. If the next cell raises an "
          "AssertionError, the repository has changed and a patch needs updating.")

expected = ["model.py", "batch.py", "loss.py", "search.py", "helpers.py", "vocabulary.py",
            "transformer_layers.py", "decoders.py", "encoders.py", "embeddings.py",
            "initialization.py", "constants.py", "Configs/Base.yaml"]
absent = [f for f in expected if not (REPO_DIR / f).exists()]
assert not absent, f"Files missing from the clone: {absent}"

## 5. Patch the repository so it works with iSign and a current Colab
Every edit is asserted (the cell stops with a clear message if the code it expects is not there) and idempotent. The patches are:

| File | Problem in the original | Fix |
|---|---|---|
| `vocabulary.py`, `helpers.py` | import `torchtext.data.Dataset`, which no longer exists — **`vocabulary.py`'s import was never removed before** | remove the imports; make the TensorBoard import optional |
| `batch.py` | decoder padding mask is derived from coordinate *values* (`!= 0`) and reshaped with a `F.pad` trick that depends on the number of frames vs. 226 | build the mask from true frame lengths, shape `[batch, 1, T, T]` (the shape the attention layers broadcast against); keep a `frame_mask` for the loss |
| `loss.py` | treats every zero as padding | `RegLoss(..., frame_mask=)` averages only over real frames (legacy behaviour kept when no mask is passed) |
| `model.py` | `get_loss_for_batch` knows nothing about the frame mask | pass `batch.frame_mask` to the loss |
| `search.py` | greedy decoding builds its padding mask from coordinate values, which wrongly blanks out zero-filled (missing) landmarks | nothing is padded while generating, so use an all-visible mask |
| `transformer_layers.py` | positional-encoding table of 200 000 positions (≈0.8 GB for the two tables) | `PE_MAX_LEN` positions |
| new `isign_data.py` | `torchtext` Field/Iterator pipeline | plain PyTorch `Dataset`, vocabulary builder, collator (returns `frame_lengths`) |

Not patched on purpose: `data.py`, `training.py`, `prediction.py` still import `torchtext`; they are only needed for the repository's own training loop (see the notes at the end).

In [ ]:
# CELL 8 — Apply the patches (idempotent; every edit is asserted).
MARK = "# [isign-patch]"


def _read(p):
    return Path(p).read_text(encoding="utf-8")


def _write(p, s):
    Path(p).write_text(s, encoding="utf-8")


def _replace_block(src, start_marker, end_marker, new_lines, what):
    """Replace the lines from the one containing start_marker through the one containing end_marker."""
    lines = src.split("\n")
    starts = [i for i, l in enumerate(lines) if start_marker in l]
    ends = [i for i, l in enumerate(lines) if end_marker in l]
    assert len(starts) == 1 and len(ends) == 1, \
        f"{what}: expected exactly one start and one end line, found {len(starts)}/{len(ends)}"
    s, e = starts[0], ends[0]
    assert 0 <= e - s <= 12, f"{what}: unexpected block size ({e - s + 1} lines)"
    return "\n".join(lines[:s] + new_lines + lines[e + 1:])


# ---------------------------------------------------------------- vocabulary.py
def patch_vocabulary(s):
    s2 = re.sub(r"^from torchtext[^\n]*\n", "", s, flags=re.M)
    assert s2 != s, "no torchtext import found"
    assert "dataset: Dataset" in s2, "build_vocab signature not found"
    return s2.replace("dataset: Dataset", "dataset")


# ---------------------------------------------------------------- helpers.py
def patch_helpers(s):
    s2 = re.sub(r"^from torchtext[^\n]*\n", "", s, flags=re.M)
    assert s2 != s, "no torchtext import found"
    old = "from torch.utils.tensorboard import SummaryWriter\n"
    assert old in s2, "tensorboard import not found"
    new = ("try:\n"
           "    from torch.utils.tensorboard import SummaryWriter\n"
           "except Exception:  # only needed for training logs\n"
           "    SummaryWriter = None\n")
    return s2.replace(old, new)


# ---------------------------------------------------------------- transformer_layers.py
def patch_transformer_layers(s):
    old = "max_len: int = 200000"
    assert s.count(old) == 1, "positional-encoding max_len not found"
    return s.replace(old, f"max_len: int = {PE_MAX_LEN}")


# ---------------------------------------------------------------- batch.py
def patch_batch(s):
    old = "        self.trg_mask = None\n"
    assert s.count(old) == 1, "trg_mask initialisation not found"
    s = s.replace(old, old + f"        self.frame_mask = None  {MARK}\n")

    new_block = [
        f"            {MARK} masks come from true frame lengths, never from coordinate values",
        "            seq_len = self.trg_input.shape[1]",
        "            if hasattr(torch_batch, \"frame_lengths\"):",
        "                input_lengths = (torch_batch.frame_lengths.cpu() - 1).clamp_min(0)",
        "                valid_steps = torch.arange(seq_len).unsqueeze(0) < input_lengths.unsqueeze(1)",
        "            else:",
        "                valid_steps = torch.ones((self.trg_input.shape[0], seq_len), dtype=torch.bool)",
        "            self.frame_mask = valid_steps  # [batch, T]: True for real frames",
        "            # decoder padding mask [batch, 1, T, T]: True where the key frame is real",
        "            self.trg_mask = valid_steps.unsqueeze(1).unsqueeze(2).expand(-1, 1, seq_len, -1).contiguous()",
    ]
    s = _replace_block(s, "Target Pad is dynamic", "self.trg_mask = (F.pad(", new_block, "batch.py mask block")

    old = "            self.trg_mask = self.trg_mask.cuda()\n"
    assert s.count(old) == 1, "_make_cuda trg_mask line not found"
    return s.replace(old, old + "            self.frame_mask = self.frame_mask.cuda()\n")


# ---------------------------------------------------------------- loss.py
NEW_REGLOSS = '''class RegLoss(nn.Module):
    """
    Regression loss. With `frame_mask` ([batch, time], True for real frames) only real
    frames contribute; without it the original behaviour (zeros are padding) is kept.
    """

    def __init__(self, cfg, target_pad=0.0):
        super(RegLoss, self).__init__()

        self.loss = cfg["training"]["loss"].lower()

        if self.loss == "l1":
            self.criterion = nn.L1Loss(reduction="none")
        elif self.loss == "mse":
            self.criterion = nn.MSELoss(reduction="none")
        else:
            print("Loss not found - revert to default L1 loss")
            self.criterion = nn.L1Loss(reduction="none")

        model_cfg = cfg["model"]

        self.target_pad = target_pad
        self.loss_scale = model_cfg.get("loss_scale", 1.0)

    # pylint: disable=arguments-differ
    def forward(self, preds, targets, frame_mask=None):

        if frame_mask is None:
            # legacy behaviour: every zero in the target is treated as padding
            loss_mask = (targets != self.target_pad).to(preds.dtype)
            loss = self.criterion(preds * loss_mask, targets * loss_mask).mean()
        else:
            if frame_mask.dim() != 2 or tuple(frame_mask.shape) != tuple(targets.shape[:2]):
                raise ValueError(
                    "frame_mask must have shape [batch, time] = %s, got %s"
                    % (tuple(targets.shape[:2]), tuple(frame_mask.shape)))
            keep = frame_mask.to(device=targets.device, dtype=preds.dtype)
            keep = keep.unsqueeze(-1).expand_as(targets)
            loss = (self.criterion(preds, targets) * keep).sum() / keep.sum().clamp_min(1.0)

        if self.loss_scale != 1.0:
            loss = loss * self.loss_scale

        return loss

'''


def patch_loss(s):
    start = s.index("class RegLoss(")
    end = s.index("class XentLoss(")
    assert start < end, "unexpected class order in loss.py"
    s = s[:start] + NEW_REGLOSS + s[end:]
    old = "from torch import nn, Tensor"
    assert old in s, "torch import line not found in loss.py"
    return s.replace(old, "import torch\n" + old, 1)


# ---------------------------------------------------------------- model.py
def patch_model(s):
    old = "        batch_loss = loss_function(skel_out, batch.trg)\n"
    assert s.count(old) == 1, "loss call in get_loss_for_batch not found"
    new = (f"        {MARK} pass the valid-frame mask so padded frames never enter the loss\n"
           "        frame_mask = getattr(batch, \"frame_mask\", None)\n"
           "        if frame_mask is not None and self.future_prediction == 0:\n"
           "            batch_loss = loss_function(skel_out, batch.trg, frame_mask=frame_mask)\n"
           "        else:\n"
           "            batch_loss = loss_function(skel_out, batch.trg)\n")
    return s.replace(old, new)


# ---------------------------------------------------------------- search.py
def patch_search(s):
    new_block = [
        f"        {MARK} nothing is padded while generating, so every generated frame is visible",
        "        padding_mask = torch.ones(",
        "            (ys.size(0), 1, ys.size(1), ys.size(1)), dtype=torch.bool, device=ys.device)",
    ]
    return _replace_block(s, "# Cut padding mask to required size", "padding_mask = (F.pad(", new_block, "search.py")


PATCHES = [
    ("vocabulary.py", patch_vocabulary),
    ("helpers.py", patch_helpers),
    ("transformer_layers.py", patch_transformer_layers),
    ("batch.py", patch_batch),
    ("loss.py", patch_loss),
    ("model.py", patch_model),
    ("search.py", patch_search),
]

for filename, fn in PATCHES:
    path = REPO_DIR / filename
    src = _read(path)
    if MARK in src:
        print(f"{filename:24s} already patched")
        continue
    try:
        new = fn(src)
    except AssertionError as exc:
        raise AssertionError(f"Patch for {filename} did not apply: {exc}") from exc
    _write(path, new.rstrip("\n") + f"\n\n{MARK} applied\n")
    print(f"{filename:24s} patched")

# ---------------------------------------------------------------- isign_data.py (new file)
ISIGN_DATA_SRC = r'''
"""
Native PyTorch data pipeline for iSign (replaces the torchtext Field/Iterator code).
Written into the Progressive Transformers repository by the notebook; import as `isign_data`.
"""
import re
from collections import Counter
from functools import partial
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

from constants import DEFAULT_UNK_ID, EOS_TOKEN, PAD_TOKEN
from vocabulary import Vocabulary

_TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize(text):
    """Lower-case and split into words and punctuation marks."""
    return _TOKEN_RE.findall(text.lower())


class ISignDataset(Dataset):
    """Reads <prefix>.text / .skels / .files, one sequence per line."""

    def __init__(self, prefix, trg_size=225, max_frames=None):
        prefix = str(prefix)

        def read(ext):
            return Path(prefix + ext).read_text(encoding="utf-8").splitlines()

        texts, skels, files = read(".text"), read(".skels"), read(".files")
        if not (len(texts) == len(skels) == len(files)):
            raise ValueError(
                f"{prefix}: text/skels/files line counts differ ({len(texts)}/{len(skels)}/{len(files)})")

        self.frame_width = trg_size + 1  # coordinates + progress counter
        self.examples = []
        for text, skel, uid in zip(texts, skels, files):
            values = np.array(skel.split(), dtype=np.float32)
            if values.size == 0 or values.size % self.frame_width:
                raise ValueError(
                    f"{uid}: {values.size} values is not a multiple of the frame width {self.frame_width}")
            if not np.isfinite(values).all():
                raise ValueError(f"{uid}: non-finite values in the skeleton data")
            frames = torch.from_numpy(values.reshape(-1, self.frame_width))
            if frames.shape[0] < 2:
                raise ValueError(f"{uid}: needs at least 2 frames (input and target are shifted by one)")
            if max_frames is not None and frames.shape[0] > max_frames:
                raise ValueError(f"{uid}: {frames.shape[0]} frames exceeds max_frames={max_frames}")
            tokens = tokenize(text)
            if not tokens:
                raise ValueError(f"{uid}: empty text")
            self.examples.append({"src": tokens, "trg": frames, "file_paths": uid})

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def build_isign_vocab(dataset, min_freq=1, max_size=None):
    """Source vocabulary (specials + words), most frequent first."""
    counter = Counter()
    for ex in dataset.examples:
        counter.update(ex["src"])
    tokens = [t for t, c in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0])) if c >= min_freq]
    if max_size is not None:
        tokens = tokens[:max_size]
    return Vocabulary(tokens=tokens)


def collate_isign(examples, vocab):
    """Pad a list of examples into the batch object the repository's Batch class expects."""
    unk_id = DEFAULT_UNK_ID()
    pad_id = vocab.stoi[PAD_TOKEN]
    eos_id = vocab.stoi[EOS_TOKEN]

    src_seqs = [
        torch.tensor([vocab.stoi.get(t, unk_id) for t in ex["src"]] + [eos_id], dtype=torch.long)
        for ex in examples
    ]
    src_lengths = torch.tensor([len(s) for s in src_seqs], dtype=torch.long)
    src = pad_sequence(src_seqs, batch_first=True, padding_value=pad_id)

    frame_lengths = torch.tensor([ex["trg"].shape[0] for ex in examples], dtype=torch.long)
    trg = pad_sequence([ex["trg"] for ex in examples], batch_first=True, padding_value=0.0)

    return SimpleNamespace(
        src=(src, src_lengths),
        trg=trg,
        frame_lengths=frame_lengths,
        file_paths=[ex["file_paths"] for ex in examples],
    )


def make_isign_loader(dataset, vocab, batch_size=2, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=partial(collate_isign, vocab=vocab),
    )
'''
_write(REPO_DIR / "isign_data.py", ISIGN_DATA_SRC.lstrip("\n"))
print(f"{'isign_data.py':24s} written")

# ---------------------------------------------------------------- syntax check of everything we touched
for filename in [f for f, _ in PATCHES] + ["isign_data.py"]:
    compile(_read(REPO_DIR / filename), filename, "exec")
print("\nAll patched files compile.")

## 6. Read poses from the split archive and build the pilot data
The four pose parts are one ZIP cut into pieces. `VirtualConcatFile` makes them look like a single seekable file, so `zipfile` can read entries without writing a combined copy to Drive.

In [ ]:
# CELL 9 — Open the split pose archive and index it by uid.
import numpy as np
from pose_format import Pose


class VirtualConcatFile(io.BufferedIOBase):
    """Seekable, read-only view of several files joined end to end."""

    def __init__(self, paths):
        self.paths = [Path(p) for p in paths]
        self.files = [open(p, "rb") for p in self.paths]
        self.sizes = [p.stat().st_size for p in self.paths]
        self.starts, total = [], 0
        for size in self.sizes:
            self.starts.append(total)
            total += size
        self.total_size = total
        self.pos = 0

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            new_pos = offset
        elif whence == io.SEEK_CUR:
            new_pos = self.pos + offset
        elif whence == io.SEEK_END:
            new_pos = self.total_size + offset
        else:
            raise ValueError("Invalid whence")
        if new_pos < 0:
            raise ValueError("Negative seek position")
        self.pos = new_pos
        return self.pos

    def read(self, size=-1):
        if self.pos >= self.total_size:
            return b""
        if size is None or size < 0:
            size = self.total_size - self.pos
        size = min(size, self.total_size - self.pos)
        chunks = []
        while size > 0:
            idx = bisect.bisect_right(self.starts, self.pos) - 1
            local_pos = self.pos - self.starts[idx]
            take = min(size, self.sizes[idx] - local_pos)
            f = self.files[idx]
            f.seek(local_pos)
            chunk = f.read(take)
            if not chunk:
                break
            chunks.append(chunk)
            self.pos += len(chunk)
            size -= len(chunk)
        return b"".join(chunks)

    def close(self):
        for f in self.files:
            f.close()
        super().close()


# Re-running this cell: close the previous handles first.
for _name in ("archive", "virtual_file"):
    try:
        globals()[_name].close()
    except Exception:
        pass

virtual_file = VirtualConcatFile([POSE_DIR / n for n in POSE_PART_NAMES])
archive = zipfile.ZipFile(virtual_file, "r")
pose_names = [n for n in archive.namelist() if n.lower().endswith(".pose")]
pose_by_stem = {PurePosixPath(n).stem: n for n in pose_names}

print(f"Archive entries: {len(archive.namelist()):,} | .pose files: {len(pose_names):,}")
csv_hits = int(df["uid"].head(200).map(lambda u: u in pose_by_stem).sum())
print(f"CSV uids with a pose file (first 200 rows): {csv_hits}/200")
assert csv_hits > 0, "No CSV uid matches a .pose filename — check the uid/filename format"

In [ ]:
# CELL 10 — Pose -> model frames: 75 landmarks x (x, y, z), x/y scaled to [-1, 1], plus a progress counter.
KEEP_IDX = None


def resolve_landmark_layout(pose):
    """Indices of the 75 kept landmarks (body, left hand, right hand), from the file header if possible."""
    reason = ""
    try:
        ranges, offset = {}, 0
        for comp in pose.header.components:
            ranges[comp.name] = (offset, offset + len(comp.points))
            offset += len(comp.points)
        body = ranges["POSE_LANDMARKS"]
        left = ranges["LEFT_HAND_LANDMARKS"]
        right = ranges["RIGHT_HAND_LANDMARKS"]
        sizes = (body[1] - body[0], left[1] - left[0], right[1] - right[0])
        if sizes == (N_BODY, N_HAND, N_HAND) and offset == pose.body.data.shape[2]:
            idx = np.r_[body[0]:body[1], left[0]:left[1], right[0]:right[1]]
            return idx, f"from file header: body {body}, left hand {left}, right hand {right}"
        reason = f"unexpected component sizes {sizes} / total {offset}"
    except Exception as exc:  # header layout not as expected
        reason = repr(exc)
    return np.r_[0:33, 501:522, 522:543], f"fallback to fixed indices ({reason})"


def pose_to_frames(pose):
    """Return (frames [T, 226], fraction of landmarks zeroed) or None if the pose has no frames."""
    global KEEP_IDX
    data = np.ma.getdata(pose.body.data)
    conf = np.ma.getdata(pose.body.confidence)
    if data.shape[0] == 0 or data.shape[1] == 0:
        return None
    if KEEP_IDX is None:
        KEEP_IDX, how = resolve_landmark_layout(pose)
        print("Landmark layout:", how)

    xyz = np.nan_to_num(data[:, 0][:, KEEP_IDX, :].astype(np.float32))   # [T, 75, 3]
    scores = np.nan_to_num(conf[:, 0][:, KEEP_IDX].astype(np.float32))   # [T, 75]

    xyz[..., 0] = xyz[..., 0] / COORD_HALF_RANGE - 1.0
    xyz[..., 1] = xyz[..., 1] / COORD_HALF_RANGE - 1.0
    missing = scores < CONF_THRESHOLD
    xyz[missing] = 0.0   # missing / low-confidence landmarks become exact zeros

    n_frames = xyz.shape[0]
    counter = np.arange(n_frames, dtype=np.float32) / n_frames
    frames = np.concatenate([xyz.reshape(n_frames, -1), counter[:, None]], axis=1)
    assert frames.shape[1] == TRG_SIZE + 1
    return frames, float(missing.mean())


print("Converter ready.")

In [ ]:
# CELL 11 — Write the pilot train / dev / test files and sanity-check the converted values.
WORK_DIR.mkdir(parents=True, exist_ok=True)
skipped = Counter()


def valid_samples(frame):
    """Yield (uid, text, frames, missing_fraction) for rows that have a usable pose, in CSV order."""
    for uid, text in zip(frame["uid"], frame["text"].astype(str)):
        text = " ".join(text.split()).lower()
        if not text:
            skipped["empty text"] += 1
            continue
        name = pose_by_stem.get(uid)
        if name is None:
            skipped["no pose file"] += 1
            continue
        try:
            converted = pose_to_frames(Pose.read(archive.read(name)))
        except Exception as exc:
            skipped[f"unreadable pose ({type(exc).__name__})"] += 1
            continue
        if converted is None:
            skipped["no frames"] += 1
            continue
        frames, missing_fraction = converted
        if not (MIN_FRAMES <= len(frames) <= MAX_FRAMES):
            skipped["length outside MIN/MAX_FRAMES"] += 1
            continue
        yield uid, text, frames, missing_fraction


def describe(split, rows):
    lengths = np.array([len(r[2]) for r in rows])
    print(f"{split:5s} | {len(rows):3d} sequences | frames min/mean/max = "
          f"{lengths.min()}/{lengths.mean():.0f}/{lengths.max()} | "
          f"landmarks zeroed: {np.mean([r[3] for r in rows]):.1%}")
    landmarks = np.concatenate([r[2][:, :-1].reshape(-1, N_LANDMARKS, 3) for r in rows])
    present = np.any(landmarks != 0, axis=-1)
    for axis, label in enumerate("xyz"):
        vals = landmarks[..., axis][present]
        print(f"        {label}: [{vals.min():+.2f}, {vals.max():+.2f}]  (present landmarks)")
    return landmarks, present


sample_stream = valid_samples(df)
all_rows = {}
for split, n in {"train": N_TRAIN, "dev": N_DEV, "test": N_TEST}.items():
    rows = list(itertools.islice(sample_stream, n))
    if len(rows) < n:
        raise RuntimeError(f"Only found {len(rows)} usable sequences for '{split}' (wanted {n}). Skipped: {dict(skipped)}")
    (WORK_DIR / f"{split}.text").write_text("\n".join(r[1] for r in rows) + "\n", encoding="utf-8")
    (WORK_DIR / f"{split}.files").write_text("\n".join(r[0] for r in rows) + "\n", encoding="utf-8")
    (WORK_DIR / f"{split}.skels").write_text(
        "\n".join(" ".join(f"{v:.5f}" for v in r[2].reshape(-1)) for r in rows) + "\n", encoding="utf-8")
    all_rows[split] = rows

print("Skipped while searching:", dict(skipped) or "nothing")
print()
landmarks, present = describe("train", all_rows["train"])
describe("dev", all_rows["dev"])
describe("test", all_rows["test"])

# Warn (don't stop) if the x/y scaling or z range looks off.
for axis, label in ((0, "x"), (1, "y")):
    vals = landmarks[..., axis][present]
    if np.abs(vals).max() > 1.5:
        print(f"\nWARNING: {label} reaches {np.abs(vals).max():.2f}; COORD_HALF_RANGE ({COORD_HALF_RANGE}) may be wrong for this data.")
zvals = landmarks[..., 2][present]
if np.abs(zvals).max() > 5:
    print(f"\nWARNING: z reaches {np.abs(zvals).max():.2f}; it is not rescaled and could dominate the loss.")
print("\nFiles written to", WORK_DIR)

## 7. Model configuration, data, and model

In [ ]:
# CELL 12 — Write the pilot configuration (Base.yaml with iSign settings).
import yaml
import torch

cfg = yaml.safe_load((REPO_DIR / "Configs" / "Base.yaml").read_text())

cfg["data"]["src"] = "text"
cfg["data"]["train"] = str(WORK_DIR / "train")
cfg["data"]["dev"] = str(WORK_DIR / "dev")
cfg["data"]["test"] = str(WORK_DIR / "test")
cfg["data"].pop("src_vocab", None)          # the vocabulary is built from the iSign text instead

cfg["model"]["trg_size"] = TRG_SIZE         # 225 coordinates (+1 counter added by build_model)

cfg["training"].update({
    "use_cuda": bool(torch.cuda.is_available()),
    "batch_size": BATCH_SIZE,
    "epochs": 1,
    "continue": False,
    "overwrite": True,
    "model_dir": str(WORK_DIR / "model"),
    "random_seed": SEED,
})

CONFIG_PATH = WORK_DIR / "config.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False))
cfg = yaml.safe_load(CONFIG_PATH.read_text())   # use exactly what was written

assert cfg["model"]["trg_size"] == TRG_SIZE
print("Config:", CONFIG_PATH)
print("  target coordinates:", cfg["model"]["trg_size"], "| frame width:", cfg["model"]["trg_size"] + 1)
print("  loss:", cfg["training"]["loss"], "| use_cuda:", cfg["training"]["use_cuda"], "| batch size:", cfg["training"]["batch_size"])

In [ ]:
# CELL 13 — Import the patched repository modules and load the pilot data.
def reset_repo_imports():
    """Drop cached copies of the repository's modules so a re-run always sees the patched files."""
    names = {p.stem for p in REPO_DIR.glob("*.py") if not p.stem.startswith("__")}
    for name in names:
        sys.modules.pop(name, None)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    importlib.invalidate_caches()


reset_repo_imports()

from constants import PAD_TOKEN, EOS_TOKEN, UNK_TOKEN
from vocabulary import Vocabulary
from batch import Batch
from loss import RegLoss
from model import build_model
import isign_data

train_set = isign_data.ISignDataset(WORK_DIR / "train", trg_size=TRG_SIZE, max_frames=MAX_FRAMES)
dev_set = isign_data.ISignDataset(WORK_DIR / "dev", trg_size=TRG_SIZE, max_frames=MAX_FRAMES)
test_set = isign_data.ISignDataset(WORK_DIR / "test", trg_size=TRG_SIZE, max_frames=MAX_FRAMES)

src_vocab = isign_data.build_isign_vocab(train_set)           # built from the training split only
vocab_path = WORK_DIR / "src_vocab.txt"
src_vocab.to_file(str(vocab_path))
assert len(Vocabulary(file=str(vocab_path))) == len(src_vocab), "vocabulary file round-trip failed"

train_loader = isign_data.make_isign_loader(train_set, src_vocab, batch_size=BATCH_SIZE, shuffle=False)
dev_loader = isign_data.make_isign_loader(dev_set, src_vocab, batch_size=BATCH_SIZE)
test_loader = isign_data.make_isign_loader(test_set, src_vocab, batch_size=BATCH_SIZE)

print(f"train/dev/test sequences: {len(train_set)}/{len(dev_set)}/{len(test_set)}")
print(f"source vocabulary (incl. 4 specials): {len(src_vocab)}")
print("example tokens:", train_set[0]["src"][:12])

In [ ]:
# CELL 14 — Build the model for 225-coordinate iSign targets.
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

trg_vocab = Vocabulary(tokens=[])   # targets are continuous vectors, not tokens
model = build_model(cfg, src_vocab=src_vocab, trg_vocab=trg_vocab)
device = torch.device("cuda" if cfg["training"]["use_cuda"] else "cpu")
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model built on {device} | parameters: {n_params / 1e6:.1f} M")
print("Source vocabulary :", len(src_vocab))
print("Target input width:", model.in_trg_size)
print("Target output width:", model.out_trg_size)
print("Decoder output layer:", model.decoder.output_layer.out_features)
print("Target embedding   :", model.trg_embed)

assert model.in_trg_size == TRG_SIZE + 1
assert model.out_trg_size == TRG_SIZE + 1
assert model.decoder.output_layer.out_features == TRG_SIZE + 1
assert model.trg_embed.in_features == TRG_SIZE + 1
assert model.src_embed.lut.num_embeddings == len(src_vocab)
print("\nModel dimension checks passed.")

### 7b. Optional: reuse weights from a checkpoint trained on another dataset
Not needed for training from scratch (the repository has no pretrained weights). If you have a checkpoint trained with a different target size or vocabulary, this loads only the tensors whose names **and** shapes match (for example the Transformer layers) and leaves the size-dependent layers (`trg_embed`, `decoder.output_layer`, `src_embed`) freshly initialised.

In [ ]:
# CELL 14b — Optional: load only the shape-compatible weights from an older checkpoint.
PRETRAINED_CKPT = None   # e.g. "/content/drive/MyDrive/ISL_Project/old_model/best.ckpt"


def load_compatible_weights(target_model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)   # only open checkpoints you trust
    state = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    own = target_model.state_dict()
    keep = {k: v for k, v in state.items() if k in own and own[k].shape == v.shape}
    skipped = sorted(k for k in state if k not in keep)
    target_model.load_state_dict(keep, strict=False)
    print(f"Loaded {len(keep)} tensors; skipped {len(skipped)} (missing or different shape):")
    for k in skipped:
        print("  -", k, tuple(state[k].shape), "->", tuple(own[k].shape) if k in own else "not in this model")


if PRETRAINED_CKPT:
    load_compatible_weights(model, PRETRAINED_CKPT)
else:
    print("No checkpoint requested — training would start from scratch.")

## 8. Checks before training
Each check stops with an assertion if something is wrong. Run them all; the last cell prints the summary.

In [ ]:
# TEST 1 — Data pipeline: shapes, the one-frame shift, masks, vocabulary ids.
pad_id = src_vocab.stoi[PAD_TOKEN]


def make_batch(examples):
    raw = isign_data.collate_isign(examples, src_vocab)
    return Batch(torch_batch=raw, pad_index=pad_id, model=model), raw


raw = next(iter(train_loader))
batch = Batch(torch_batch=raw, pad_index=pad_id, model=model)

B, T_raw = raw.trg.shape[0], raw.trg.shape[1]
L = T_raw - 1
print("Raw target:", tuple(raw.trg.shape), "| frame lengths:", raw.frame_lengths.tolist())
print("trg_input :", tuple(batch.trg_input.shape), "| trg:", tuple(batch.trg.shape))
print("trg_mask  :", tuple(batch.trg_mask.shape), batch.trg_mask.dtype)

assert raw.trg.shape[-1] == TRG_SIZE + 1
assert tuple(batch.trg_input.shape) == (B, L, TRG_SIZE + 1) == tuple(batch.trg.shape)
assert torch.equal(batch.trg[:, :-1].cpu(), batch.trg_input[:, 1:].cpu()), "target is not input shifted by one frame"
assert tuple(batch.trg_mask.shape) == (B, 1, L, L) and batch.trg_mask.dtype == torch.bool
assert batch.frame_mask.sum(1).cpu().tolist() == (raw.frame_lengths - 1).tolist(), "frame mask != true lengths - 1"
assert batch.src.max().item() < len(src_vocab)

# dev/test use the training vocabulary: unseen words must map to <unk>, never out of range
for name, loader in (("dev", dev_loader), ("test", test_loader)):
    other = next(iter(loader))
    ids = other.src[0]
    assert ids.max().item() < len(src_vocab)
    unk_rate = (ids == src_vocab.stoi[UNK_TOKEN]).float().mean().item()
    print(f"{name}: source ids in range | unknown-word rate in first batch: {unk_rate:.0%}")
print("TEST 1 passed.")

In [ ]:
# TEST 2 — Teacher-forced forward pass (eval mode).
model.eval()
with torch.no_grad():
    prediction, _ = model(
        src=batch.src,
        trg_input=batch.trg_input,
        src_mask=batch.src_mask,
        src_lengths=batch.src_lengths,
        trg_mask=batch.trg_mask,
    )

print("Prediction:", tuple(prediction.shape), "| target:", tuple(batch.trg.shape))
assert prediction.shape == batch.trg.shape
assert torch.isfinite(prediction).all(), "non-finite predictions"
assert torch.isfinite(batch.trg).all(), "non-finite targets"
print("TEST 2 passed.")

In [ ]:
# TEST 3 — Masked loss in isolation (valid zero coordinates count, padded frames do not).
criterion = RegLoss(cfg)

preds = torch.tensor([[[1.0, 2.0], [3.0, 4.0], [100.0, 100.0]]], requires_grad=True)
targets = torch.tensor([[[0.0, 2.0], [3.0, 0.0], [0.0, 0.0]]])
frame_mask = torch.tensor([[True, True, False]])

value = criterion(preds, targets, frame_mask=frame_mask)
value.backward()

kind = cfg["training"]["loss"].lower()
expected = {"mse": 17.0 / 4, "l1": 5.0 / 4}[kind] * criterion.loss_scale   # 4 valid values: errors 1,0,0,4
print(f"Masked {kind} loss: {value.item():.4f} (expected {expected:.4f})")
print("Gradient on the padded frame:", preds.grad[0, 2].tolist())

assert abs(value.item() - expected) < 1e-5
assert torch.isfinite(preds.grad).all()
assert torch.all(preds.grad[0, 2] == 0), "padded frame received gradient"
assert preds.grad[0, 0, 0] != 0, "a valid zero-coordinate target must still produce gradient"
assert torch.isfinite(criterion(torch.ones(1, 3, 2), torch.zeros(1, 3, 2))), "legacy (unmasked) call broke"
try:
    criterion(preds, targets, frame_mask=torch.ones(1, 5, dtype=torch.bool))
    raise AssertionError("a wrongly shaped frame_mask should have been rejected")
except ValueError:
    pass
print("TEST 3 passed.")

In [ ]:
# TEST 4 — The model's own get_loss_for_batch uses the frame mask.
model.eval()
with torch.no_grad():
    loss_a, _ = model.get_loss_for_batch(batch, criterion)
    pred, _ = model(src=batch.src, trg_input=batch.trg_input, src_mask=batch.src_mask,
                    src_lengths=batch.src_lengths, trg_mask=batch.trg_mask)
    loss_b = criterion(pred, batch.trg, frame_mask=batch.frame_mask)
    loss_legacy = criterion(pred, batch.trg)

print(f"get_loss_for_batch: {loss_a.item():.6f} | explicit frame mask: {loss_b.item():.6f} | legacy zero-as-padding: {loss_legacy.item():.6f}")
assert torch.allclose(loss_a, loss_b), "get_loss_for_batch is not using the frame mask"
print("TEST 4 passed.")

In [ ]:
# TEST 5 — Forward/backward through the whole model: finite loss, finite gradients, every parameter reached.
model.train()
model.zero_grad(set_to_none=True)

batch, raw = make_batch([train_set[i] for i in range(min(BATCH_SIZE, len(train_set)))])
loss_value, _ = model.get_loss_for_batch(batch, criterion)
loss_value.backward()

no_grad = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
all_finite = all(torch.isfinite(p.grad).all().item() for p in model.parameters() if p.grad is not None)
grad_norm = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters() if p.grad is not None)).item()

print(f"Loss: {loss_value.item():.4f} | gradient norm: {grad_norm:.4f}")
if no_grad:
    print("WARNING: parameters that received no gradient:", no_grad)
assert torch.isfinite(loss_value)
assert all_finite, "non-finite gradient"
assert grad_norm > 0
model.zero_grad(set_to_none=True)
model.eval()
print("TEST 5 passed.")

In [ ]:
# TEST 6 — Padding invariance: a sequence must get the same predictions alone and padded inside a batch.
order = sorted(range(len(train_set)), key=lambda i: train_set[i]["trg"].shape[0])
pair = [train_set[order[0]], train_set[order[-1]]]    # shortest and longest -> real padding in the batch


def teacher_forced(examples):
    b, r = make_batch(examples)
    with torch.no_grad():
        out, _ = model(src=b.src, trg_input=b.trg_input, src_mask=b.src_mask,
                       src_lengths=b.src_lengths, trg_mask=b.trg_mask)
    return out.cpu(), r


together, raw_pair = teacher_forced(pair)
print("Frame lengths in the batch:", raw_pair.frame_lengths.tolist(),
      "| source lengths:", raw_pair.src[1].tolist())
worst = 0.0
for i, example in enumerate(pair):
    alone, raw_single = teacher_forced([example])
    n = int(raw_single.frame_lengths[0]) - 1
    diff = (together[i, :n] - alone[0, :n]).abs().max().item()
    worst = max(worst, diff)
    assert diff < 1e-3, f"sequence {i}: padded batch differs from single run by {diff:.2e}"
print(f"Largest difference between padded and single runs: {worst:.2e}")
print("TEST 6 passed.")

In [ ]:
# TEST 7 — Greedy decoding smoke test (the path used after training), on a short prefix to keep it fast.
batch, raw = make_batch([train_set[i] for i in range(min(BATCH_SIZE, len(train_set)))])
SMOKE_FRAMES = 40
batch.trg_input = batch.trg_input[:, :SMOKE_FRAMES].contiguous()
steps = batch.trg_input.shape[1]

model.eval()
with torch.no_grad():
    generated, _ = model.run_batch(batch, max_output_length=None)

print("Generated:", tuple(generated.shape))
assert tuple(generated.shape) == (batch.trg_input.shape[0], steps + 1, TRG_SIZE + 1)
assert torch.isfinite(generated).all(), "non-finite generated frames"
print("TEST 7 passed.")

## 9. Save the patched code and pilot data to Drive (optional) and summarise

In [ ]:
# CELL 15 — Keep the patched repository, configuration and pilot data on Drive for the training session.
if SAVE_TO_DRIVE:
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(REPO_DIR, DRIVE_SAVE_DIR / "ProgressiveTransformersSLP_patched", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns(".git", "__pycache__"))
    shutil.copytree(WORK_DIR, DRIVE_SAVE_DIR / "isl_phase2_pilot", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("model"))
    print("Saved to", DRIVE_SAVE_DIR)
else:
    print("SAVE_TO_DRIVE is False — nothing copied (Colab's local disk is erased when the runtime ends).")

print("""
READY FOR TRAINING
  data      : train/dev/test pilot in %s  (226 values per frame)
  model     : %.1f M parameters, target width %d, source vocabulary %d
  checks    : tests 1-7 passed (shapes, masks, loss, gradients, padding invariance, greedy decoding)
  not done  : no training loop, no pretrained weights (none exist for this repository)
""" % (WORK_DIR, n_params / 1e6, TRG_SIZE + 1, len(src_vocab)))

## Notes for the training step
* **Training loop.** The repository's own `training.py`, `data.py` and `prediction.py` still import `torchtext` and build torchtext iterators, so they will not import on a current Colab. Write a small loop around what is verified here: `isign_data.make_isign_loader` → `Batch(torch_batch=..., pad_index=..., model=model)` → `model.get_loss_for_batch(batch, RegLoss(cfg))` → `backward()` / optimizer step.
* **Full-dataset data format.** The `.skels` text format is fine for a pilot but far too large for all of the dataset's segments (the CSV cell above prints the exact count; each frame is 226 numbers written as text). For the full run, convert each sequence once to a `float16` `.npy` file (or a few sharded `.pt` files) and make `ISignDataset` load those.
* **Missing landmarks.** Low-confidence landmarks are stored as exact zeros, which is also a valid position (the image centre), and the loss does train the model to output them. If predictions look unstable around hands, consider excluding zeroed landmarks from the loss or adding a visibility flag.
* **Scaling of x/y/z.** `COORD_HALF_RANGE = 150` assumes pixel coordinates up to ≈300; cell 11 prints the observed ranges and warns if they do not fit. `z` is left as is.
* **Changing the pilot size.** Edit `N_TRAIN / N_DEV / N_TEST` in cell 1 and rerun from cell 11 onward (then 12–14 and the checks).